## 2. Imports and reproducibility

In [3]:

# Cell 1 — Imports and environment reproducibility

import importlib.metadata
import importlib.util
import sys
from pathlib import Path


def locate_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected notebooks/ and requirements.txt."
    )


PROJECT_ROOT = locate_project_root()

EXPECTED_PYTHON = (
    PROJECT_ROOT / ".venv" / "bin" / "python"
).resolve()

ACTIVE_PYTHON = Path(sys.executable).resolve()

# Enforce the project's virtual environment.
if not EXPECTED_PYTHON.exists():
    raise RuntimeError(
        "Project virtual environment not found:\n"
        f"{EXPECTED_PYTHON}\n"
        "Create the .venv environment before continuing."
    )

if ACTIVE_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        "Incorrect notebook kernel.\n"
        f"Active: {ACTIVE_PYTHON}\n"
        f"Expected: {EXPECTED_PYTHON}\n"
        "In VS Code, select the project's .venv kernel."
    )


REQUIRED_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "pyarrow": "pyarrow",
    "joblib": "joblib",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scikit-learn": "sklearn",
    "xgboost": "xgboost",
    "IPython": "IPython",
    "ipykernel": "ipykernel",
}

missing = [
    package
    for package, module in REQUIRED_IMPORTS.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    raise ModuleNotFoundError(
        "Missing packages: " + ", ".join(missing)
    )


print("Environment preflight: PASSED")
print("Project root:", PROJECT_ROOT)
print("Python:", ACTIVE_PYTHON)

for package in REQUIRED_IMPORTS:
    print(
        f"  {package}: "
        f"{importlib.metadata.version(package)}"
    )

Environment preflight: PASSED
Project root: /home/snow/Documents/Researches/Normal vs Abnormal EEG records using ML
Python: /usr/bin/python3.12
  numpy: 2.5.4
  pandas: 3.0.6
  scipy: 1.18.1
  pyarrow: 26.0.0
  joblib: 1.6.0
  matplotlib: 3.11.2
  seaborn: 0.13.2
  scikit-learn: 1.9.1
  xgboost: 3.4.1
  IPython: 9.17.1
  ipykernel: 7.4.0


In [4]:
import json
import os
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from IPython.display import display

from sklearn.base import clone
from sklearn.feature_selection import (
    SelectKBest,
    SelectFromModel,
    RFE,
    f_classif,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# Reproducibility
RANDOM_STATE = 42

# Nested cross-validation
N_OUTER_SPLITS = 5
N_INNER_SPLITS = 5

# Safety: final evaluation remains locked
RUN_FINAL_EVALUATION = False

# Local CPU configuration
N_JOBS = max(
    1,
    min(4, (os.cpu_count() or 2) - 1)
)

# Random seed
np.random.seed(RANDOM_STATE)

# Pandas
pd.set_option("display.max_columns", 120)

# Visualization
sns.set_theme(
    style="whitegrid",
    context="notebook"
)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Feature-selection methods
SELECTION_METHODS = [
    "Baseline",
    "SelectKBest",
    "L1",
    "RFE",
]

# Experiment settings
FEATURE_COUNTS = [50, 100, 200, 500]
L1_C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0]

print("E1 Feature Selection — Configuration")
print("-" * 45)
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("Outer CV folds:", N_OUTER_SPLITS)
print("Inner CV folds:", N_INNER_SPLITS)
print("Random seed:", RANDOM_STATE)
print("CPU workers:", N_JOBS)
print("Final evaluation enabled:", RUN_FINAL_EVALUATION)

E1 Feature Selection — Configuration
---------------------------------------------
Python: 3.12.3
scikit-learn: 1.9.1
Outer CV folds: 5
Inner CV folds: 5
Random seed: 42
CPU workers: 4
Final evaluation enabled: False


## 3. Configuration and project paths

In [6]:
PROJECT_ROOT = PROJECT_ROOT.resolve()

# Input feature tables
FEATURE_ROOT = PROJECT_ROOT / "tuab_feature_extraction" / "features"

FE_ALL_PATH = FEATURE_ROOT / "tuab_recording_features.parquet"
FE_QC_PATH = FEATURE_ROOT / "tuab_recording_features_qc.parquet"

# Separate E1 outputs from existing ML baseline results
REPORT_ROOT = PROJECT_ROOT / "reports" / "feature_selection"
FIGURE_DIR = REPORT_ROOT / "figures"

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

RUN_DIR = REPORT_ROOT / "runs" / RUN_ID

OUTPUT_DIRS = {
    "integrity": RUN_DIR / "integrity",
    "nested_cv": RUN_DIR / "nested_cv",
    "selection": RUN_DIR / "selected_features",
    "stability": RUN_DIR / "stability",
    "comparisons": RUN_DIR / "comparisons",
    "models": RUN_DIR / "models",
}

# Check input files before creating output directories
assert PROJECT_ROOT.is_dir(), (
    f"Missing project directory: {PROJECT_ROOT}"
)

assert FE_ALL_PATH.is_file(), (
    f"Missing FE-All table: {FE_ALL_PATH}"
)

assert FE_QC_PATH.is_file(), (
    f"Missing FE-QC table: {FE_QC_PATH}"
)

assert not RUN_DIR.exists(), (
    f"Run directory already exists: {RUN_DIR}"
)

# Create E1 output directories
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

for directory in OUTPUT_DIRS.values():
    directory.mkdir(parents=True, exist_ok=False)

print("E1 paths: VERIFIED")
print("-" * 50)
print("Project root:", PROJECT_ROOT)
print("FE-All:", FE_ALL_PATH)
print("FE-QC:", FE_QC_PATH)
print("Run ID:", RUN_ID)
print("Run directory:", RUN_DIR)
print("CPU jobs:", N_JOBS)
print("Final evaluation enabled:", RUN_FINAL_EVALUATION)

E1 paths: VERIFIED
--------------------------------------------------
Project root: /home/snow/Documents/Researches/Normal vs Abnormal EEG records using ML
FE-All: /home/snow/Documents/Researches/Normal vs Abnormal EEG records using ML/tuab_feature_extraction/features/tuab_recording_features.parquet
FE-QC: /home/snow/Documents/Researches/Normal vs Abnormal EEG records using ML/tuab_feature_extraction/features/tuab_recording_features_qc.parquet
Run ID: 20261024T094237_544427Z
Run directory: /home/snow/Documents/Researches/Normal vs Abnormal EEG records using ML/reports/feature_selection/runs/20261024T094237_544427Z
CPU jobs: 4
Final evaluation enabled: False


## 4. Inspect FE-All and FE-QC

In [7]:
fe_all = pd.read_parquet(FE_ALL_PATH)
fe_qc = pd.read_parquet(FE_QC_PATH)

print("=" * 60)
print("1. DATASET SHAPES")
print("=" * 60)

print("FE-All:", fe_all.shape)
print("FE-QC: ", fe_qc.shape)

print("\n" + "=" * 60)
print("2. COLUMN NAMES")
print("=" * 60)

print("\nFE-All — first 25 columns:")
print(fe_all.columns[:25].tolist())

print("\nFE-QC — first 25 columns:")
print(fe_qc.columns[:25].tolist())

print("\nFE-All — last 15 columns:")
print(fe_all.columns[-15:].tolist())

print("\nFE-QC — last 15 columns:")
print(fe_qc.columns[-15:].tolist())

print("\n" + "=" * 60)
print("3. METADATA CANDIDATES")
print("=" * 60)

keywords = [
    "patient", "subject", "label",
    "target", "split", "record",
    "epoch", "qc", "accepted",
    "path", "file"
]

for name, df in [("FE-All", fe_all), ("FE-QC", fe_qc)]:
    metadata = [
        col for col in df.columns
        if any(word in col.lower() for word in keywords)
    ]

    print(f"\n{name} — possible metadata columns:")
    print(metadata[:60])

print("\n" + "=" * 60)
print("4. MISSING VALUES")
print("=" * 60)

for name, df in [("FE-All", fe_all), ("FE-QC", fe_qc)]:
    missing_rows = df.isna().any(axis=1).sum()
    missing_columns = df.isna().any(axis=0).sum()

    print(f"\n{name}:")
    print("Rows containing NaN:", missing_rows)
    print("Columns containing NaN:", missing_columns)

print("\nSchema inspection: COMPLETE")

1. DATASET SHAPES
FE-All: (2993, 1156)
FE-QC:  (2993, 1159)

2. COLUMN NAMES

FE-All — first 25 columns:
['filepath', 'patient_id', 'label', 'split', 'car__C3__alpha_absolute_power__iqr', 'car__C3__alpha_absolute_power__median', 'car__C3__alpha_relative_power__iqr', 'car__C3__alpha_relative_power__median', 'car__C3__beta_absolute_power__iqr', 'car__C3__beta_absolute_power__median', 'car__C3__beta_relative_power__iqr', 'car__C3__beta_relative_power__median', 'car__C3__delta_absolute_power__iqr', 'car__C3__delta_absolute_power__median', 'car__C3__delta_alpha_ratio__iqr', 'car__C3__delta_alpha_ratio__median', 'car__C3__delta_relative_power__iqr', 'car__C3__delta_relative_power__median', 'car__C3__hjorth_complexity__iqr', 'car__C3__hjorth_complexity__median', 'car__C3__hjorth_mobility__iqr', 'car__C3__hjorth_mobility__median', 'car__C3__iqr__iqr', 'car__C3__iqr__median', 'car__C3__line_length__iqr']

FE-QC — first 25 columns:
['filepath', 'patient_id', 'label', 'split', 'total_epochs', 'ac

## 5. Dataset integrity and QC eligibility

In [ ]:
METADATA_COLS = [
    "filepath", "patient_id", "label", "split"
]

QC_COLS = [
    "total_epochs",
    "accepted_epochs",
    "accepted_fraction"
]

FEATURE_COLS = [
    col for col in fe_all.columns
    if col not in METADATA_COLS
]

# 1. Verify schemas and feature alignment
assert all(col in fe_all.columns for col in METADATA_COLS)
assert all(col in fe_qc.columns for col in METADATA_COLS + QC_COLS)

assert len(FEATURE_COLS) == 1152
assert len(FEATURE_COLS) == len(set(FEATURE_COLS))

assert fe_qc.columns.tolist() == (
    METADATA_COLS + QC_COLS + FEATURE_COLS
), "FE-QC feature columns or ordering differ from FE-All."

print("1. Feature schema: PASSED")
print("EEG feature count:", len(FEATURE_COLS))

# 2. Verify recording identity
for name, df in [("FE-All", fe_all), ("FE-QC", fe_qc)]:
    assert df["filepath"].notna().all()
    assert df["patient_id"].notna().all()
    assert df["label"].notna().all()
    assert df["split"].notna().all()
    assert df["filepath"].is_unique, f"Duplicate filepath in {name}"

all_metadata = fe_all[METADATA_COLS].sort_values(
    "filepath"
).reset_index(drop=True)

qc_metadata = fe_qc[METADATA_COLS].sort_values(
    "filepath"
).reset_index(drop=True)

pd.testing.assert_frame_equal(
    all_metadata,
    qc_metadata,
    check_dtype=False
)

print("\n2. Recording identity: PASSED")

# 3. QC eligibility
qc_eligible = fe_qc["accepted_epochs"].gt(0)

print("\n3. QC eligibility")
print("Eligible recordings:", int(qc_eligible.sum()))
print("Ineligible recordings:", int((~qc_eligible).sum()))

print("\nAccepted epochs summary:")
print(fe_qc["accepted_epochs"].describe())

# 4. Missing EEG features by QC eligibility
feature_missing = fe_qc[FEATURE_COLS].isna().any(axis=1)

print("\n4. Missing-feature consistency")
print(pd.crosstab(
    qc_eligible.rename("QC eligible"),
    feature_missing.rename("Has missing EEG features")
))

# 5. Split distribution
print("\n5. FE-QC split distribution")
print(pd.crosstab(
    fe_qc["split"],
    fe_qc["label"],
    margins=True
))

print("\n6. Eligible FE-QC split distribution")
print(pd.crosstab(
    fe_qc.loc[qc_eligible, "split"],
    fe_qc.loc[qc_eligible, "label"],
    margins=True
))

# 7. Patient independence
patient_sets = {
    split: set(group["patient_id"])
    for split, group in fe_qc.groupby("split")
}

print("\n7. Patient independence")

split_names = list(patient_sets)

for i, split_a in enumerate(split_names):
    for split_b in split_names[i + 1:]:
        overlap = patient_sets[split_a] & patient_sets[split_b]
        print(f"{split_a} vs {split_b}: {len(overlap)} overlapping patients")
        assert not overlap, "Patient leakage detected!"

print("\nCell 5 integrity checks: PASSED")

1. Feature schema: PASSED
EEG feature count: 1152

2. Recording identity: PASSED

3. QC eligibility
Eligible recordings: 2979
Ineligible recordings: 14

Accepted epochs summary:
count    2993.000000
mean      152.882392
std        59.187815
min         0.000000
25%       140.000000
50%       150.000000
75%       164.000000
max      1731.000000
Name: accepted_epochs, dtype: float64

4. Missing-feature consistency
Has missing EEG features  False  True 
QC eligible                           
False                         0     14
True                       2979      0

5. FE-QC split distribution
label  abnormal  normal   All
split                        
eval        126     150   276
train      1076    1097  2173
val         270     274   544
All        1472    1521  2993

6. Eligible FE-QC split distribution
label  abnormal  normal   All
split                        
eval        126     150   276
train      1068    1097  2165
val         266     272   538
All        1460    1519  2979



## 6. Prepare the E1 development dataset

In [9]:
# Keep only QC-eligible training recordings
e1_train = fe_qc.loc[
    (fe_qc["split"] == "train") &
    (fe_qc["accepted_epochs"] > 0)
].copy()

# Feature matrix: 1,152 EEG features
X_train = e1_train[FEATURE_COLS].copy()

# Binary labels
y_train = e1_train["label"].map({
    "normal": 0,
    "abnormal": 1,
})

# Patient grouping
groups_train = e1_train["patient_id"].copy()

# Integrity checks
assert X_train.shape == (2165, 1152)
assert y_train.notna().all()
assert set(y_train.unique()) == {0, 1}
assert groups_train.notna().all()
assert not X_train.isna().any().any()
assert np.isfinite(X_train.to_numpy()).all()

# Keep row indices aligned
X_train = X_train.reset_index(drop=True)
y_train = y_train.astype(int).reset_index(drop=True)
groups_train = groups_train.reset_index(drop=True)

print("E1 training data: READY")
print("-" * 45)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Patient groups:", groups_train.nunique())

print("\nClass distribution:")
print(y_train.value_counts().rename(index={
    0: "normal",
    1: "abnormal",
}))

print("\nFeature count:", len(FEATURE_COLS))
print("Validation/evaluation data used: NO")

E1 training data: READY
---------------------------------------------
X_train: (2165, 1152)
y_train: (2165,)
Patient groups: 1662

Class distribution:
label
normal      1097
abnormal    1068
Name: count, dtype: int64

Feature count: 1152
Validation/evaluation data used: NO


## 7. Verify nested CV feasibility

In [10]:
outer_cv = StratifiedGroupKFold(
    n_splits=N_OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

inner_cv = StratifiedGroupKFold(
    n_splits=N_INNER_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE + 1,
)

outer_splits = list(
    outer_cv.split(X_train, y_train, groups_train)
)

for outer_id, (train_idx, test_idx) in enumerate(
    outer_splits, start=1
):
    outer_train_patients = set(groups_train.iloc[train_idx])
    outer_test_patients = set(groups_train.iloc[test_idx])

    assert outer_train_patients.isdisjoint(outer_test_patients)
    assert y_train.iloc[train_idx].nunique() == 2
    assert y_train.iloc[test_idx].nunique() == 2

    X_outer = X_train.iloc[train_idx]
    y_outer = y_train.iloc[train_idx]
    groups_outer = groups_train.iloc[train_idx]

    inner_splits = list(
        inner_cv.split(X_outer, y_outer, groups_outer)
    )

    for inner_train_idx, inner_val_idx in inner_splits:
        inner_train_patients = set(
            groups_outer.iloc[inner_train_idx]
        )
        inner_val_patients = set(
            groups_outer.iloc[inner_val_idx]
        )

        assert inner_train_patients.isdisjoint(
            inner_val_patients
        )
        assert y_outer.iloc[inner_train_idx].nunique() == 2
        assert y_outer.iloc[inner_val_idx].nunique() == 2

    print(
        f"Outer fold {outer_id}: "
        f"train={len(train_idx)}, "
        f"test={len(test_idx)}, "
        f"inner folds={len(inner_splits)}, "
        f"patient overlap=0"
    )

print("\nNested CV structure: VERIFIED")
print("Model training performed: NO")


Outer fold 1: train=1731, test=434, inner folds=5, patient overlap=0
Outer fold 2: train=1731, test=434, inner folds=5, patient overlap=0
Outer fold 3: train=1733, test=432, inner folds=5, patient overlap=0
Outer fold 4: train=1733, test=432, inner folds=5, patient overlap=0
Outer fold 5: train=1732, test=433, inner folds=5, patient overlap=0

Nested CV structure: VERIFIED
Model training performed: NO
